# Lab 03 · Reference solution

The polished final implementation of [Lab 03: Multi-step research agent](../README.md).

Two tools (`web_search`, `fetch_page`), structured-error envelope on
every failure mode (timeout, paywall, rate-limit, empty), agent loop
with action-hash deduplication and citation tracking that records URLs
*the agent actually fetched* (not URLs it merely saw in snippets).

> ⏱ Read time: ~7 min · Notebook ~20 cells.
> 📖 The lab notebook walks through three failure modes deliberately
> (empty results, paywall, repeated action). This solution ships the
> hardening as the default behavior — read the lab to see the failure
> modes in action; come here for the consolidated implementation.

## Setup

`ddgs` ships with this lab (no API key); `requests` + `bs4` for fetching;
the chat client is provider-agnostic.

In [ ]:
import hashlib
import json
import os
import pathlib
import re
import time
import warnings
from dataclasses import dataclass, field
from typing import Any, Literal

from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")

PROVIDER = "openai"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


## Provider-agnostic chat client

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(
    messages: list[dict],
    tools: list[dict] | None = None,
    tool_choice: str = "auto",
) -> AssistantMessage:
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice=tool_choice if tools else None, temperature=0,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments))
                for tc in (msg.tool_calls or [])
            ],
        )
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools or None, max_tokens=1024, temperature=0,
        )
        text, tcs = "", []
        for block in resp.content:
            if block.type == "text":
                text += block.text
            elif block.type == "tool_use":
                tcs.append(ToolCall(id=block.id, name=block.name, arguments=dict(block.input)))
        return AssistantMessage(content=text or None, tool_calls=tcs)
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


## `web_search`

Wraps `ddgs` with: a structured `recency` enum (mapped to ddgs's
timelimit codes), normalized result shape (title/url/snippet), and
structured-error envelope on every documented failure mode. The
normalized shape means swapping to Tavily later is a one-function
change.

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException

RecencyType = Literal["any", "day", "week", "month", "year"]

_RECENCY_MAP: dict[str, str | None] = {
    "any": None, "day": "d", "week": "w", "month": "m", "year": "y",
}


def web_search(
    query: str,
    recency: RecencyType = "any",
    max_results: int = 8,
) -> dict:
    """Search the web. Returns structured results or a structured error."""
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}

    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(
                query=query.strip(), region="us-en", safesearch="moderate",
                timelimit=_RECENCY_MAP.get(recency),
                max_results=max_results, backend="auto",
            )
    except RatelimitException as e:
        return {"status": "error", "kind": "rate_limit", "detail": str(e)}
    except TimeoutException as e:
        return {"status": "error", "kind": "timeout", "detail": str(e)}
    except DDGSException as e:
        return {"status": "error", "kind": "other", "detail": str(e)}
    except Exception as e:
        return {"status": "error", "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}

    if not raw:
        return {"status": "empty", "query": query, "detail": "no results returned"}

    normalized = [
        {
            "title": (r.get("title") or "").strip(),
            "url": (r.get("href") or "").strip(),
            "snippet": (r.get("body") or "").strip(),
        }
        for r in raw
        if r.get("href")
    ]
    return {"status": "ok", "results": normalized[:max_results]}


## `fetch_page`

Distinguishes the four failure modes that matter for an agent: timeout,
4xx (blocked vs http_4xx), 5xx, paywall (detected via body heuristics),
and `too_long` (truncated but readable). Identifies itself via a custom
User-Agent so site operators can block if they choose.

In [ ]:
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

USER_AGENT = (
    "AgenticAIEngineer-CourseLab/0.1 "
    "(https://github.com/MHHamdan/Agentic-AI-Engineer) "
    "Mozilla/5.0 (compatible)"
)

PAYWALL_MARKERS = [
    "subscribe to read", "subscribe to continue",
    "create a free account to continue",
    "you've reached your free article limit",
    "register to read",
]


def fetch_page(url: str, max_chars: int = 8000) -> dict:
    """Fetch a URL; return cleaned text or structured error."""
    if not url or not url.startswith(("http://", "https://")):
        return {"status": "error", "url": url, "kind": "other", "detail": "invalid url"}

    t0 = time.monotonic()
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT},
                            timeout=15, allow_redirects=True)
    except requests.Timeout:
        return {"status": "error", "url": url, "kind": "timeout",
                "detail": "request timed out after 15s"}
    except requests.ConnectionError as e:
        return {"status": "error", "url": url, "kind": "other",
                "detail": f"connection error: {e}"}
    except requests.RequestException as e:
        return {"status": "error", "url": url, "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}

    elapsed_ms = int((time.monotonic() - t0) * 1000)

    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return {"status": "error", "url": url, "kind": kind,
                "detail": f"HTTP {resp.status_code}"}
    if 500 <= resp.status_code < 600:
        return {"status": "error", "url": url, "kind": "http_5xx",
                "detail": f"HTTP {resp.status_code}"}

    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return {"status": "error", "url": url, "kind": "parse",
                "detail": f"{type(e).__name__}: {e}"}

    for tag in soup(["script", "style", "nav", "footer", "aside",
                     "header", "form", "iframe", "noscript"]):
        tag.decompose()

    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n")).strip()

    body_lower = text[:5000].lower()
    if any(marker in body_lower for marker in PAYWALL_MARKERS):
        return {"status": "error", "url": url, "kind": "paywall",
                "detail": "paywall markers detected in body"}

    if len(text) > max_chars:
        return {"status": "too_long", "url": url, "title": title,
                "text": text[:max_chars], "truncated_at": max_chars,
                "total_chars": len(text), "elapsed_ms": elapsed_ms}

    return {"status": "ok", "url": url, "title": title, "text": text,
            "elapsed_ms": elapsed_ms}


## Tool schemas + dispatcher

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": (
                "Search the web. Returns up to max_results items with title, url, "
                "and a short snippet. Use this to triage which pages are worth "
                "fetching. Use recency='week' or 'month' for news; leave as 'any' "
                "for stable knowledge."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query, 3-8 words usually best."},
                    "recency": {
                        "type": "string",
                        "enum": ["any", "day", "week", "month", "year"],
                        "description": "Time scope. 'any' for stable topics; 'week'/'month' for news; 'day' for breaking.",
                    },
                    "max_results": {"type": "integer", "description": "1-10, default 8."},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_page",
            "description": (
                "Fetch the full content of a single URL. Use when a snippet isn't "
                "enough to answer. Returns cleaned text; may return a structured "
                "error (timeout, http_4xx, blocked, paywall) or 'too_long' with "
                "truncation. Do NOT call repeatedly on the same URL."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "Absolute https:// or http:// URL."},
                    "max_chars": {"type": "integer", "description": "Truncate body to this length. Default 8000."},
                },
                "required": ["url"],
            },
        },
    },
]


def execute_tool(name: str, args: dict) -> dict:
    """Dispatch a tool call to its Python handler."""
    if name == "web_search":
        return web_search(query=args["query"],
                          recency=args.get("recency", "any"),
                          max_results=args.get("max_results", 8))
    if name == "fetch_page":
        return fetch_page(url=args["url"],
                          max_chars=args.get("max_chars", 8000))
    return {"status": "error", "kind": "other", "detail": f"unknown tool: {name}"}


## The agent loop

Three properties to highlight:

1. **Citations track what was *fetched*, not what was *seen*.** Only successful `fetch_page` results (status `ok` or `too_long`) get appended. The lab framework here is by-the-loop, not by-the-LLM — the model can't fabricate citations.
2. **Action hash dedup** on `(name, sorted_args)` prevents the classic infinite-loop where the model retries the same empty-result query.
3. **Tool results are capped at 4000 chars** when appended to state to keep context manageable across multi-step trajectories.

In [ ]:
MAX_STEPS = 8

SYSTEM_PROMPT = """You are a research assistant. Answer the user's question
by searching the web and reading relevant pages.

Strategy:
1. Start with a web_search using 3-8 specific words from the question.
2. Look at the snippets. If they answer the question, synthesize directly.
   If not, pick the 1-2 most relevant URLs and call fetch_page on them.
3. After reading, if you still need more, search again with a refined query
   — but do not repeat queries you've already tried.
4. Stop when you can answer confidently with grounded evidence, OR after
   you have enough evidence that further searches won't help. Don't loop.
5. In your final answer, cite the URLs you actually read. Do not cite URLs
   you only saw in search results but did not open.

When you cannot find a confident answer, say so plainly. Do not guess or
fabricate facts. "I could not find a reliable answer" is acceptable.

Use 'recency' to scope by freshness: 'week' or 'month' for current events,
'any' for stable knowledge.
"""


def _action_hash(name: str, args: dict) -> str:
    """Deterministic hash of a (tool_name, sorted_args) pair for dedup."""
    payload = name + "|" + json.dumps(args, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def run_agent(question: str, max_steps: int = MAX_STEPS, verbose: bool = True) -> dict:
    """Run the research agent. Returns answer + citations + trace metadata."""
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n── Step {step} ──")

        msg = chat_with_tools(messages, tools=TOOLS)

        assistant_entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            assistant_entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_entry)

        if not msg.tool_calls:
            if verbose:
                print(f"  ◆ FINAL: {(msg.content or '')[:120]}...")
            return {
                "answer": msg.content or "",
                "citations": citations,
                "steps": step,
                "stopped_reason": "answer_with_citations" if citations else "answer_without_fetch",
            }

        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen_actions:
                tool_result = {
                    "status": "error", "kind": "repeated_action",
                    "detail": f"You already called {tc.name} with these arguments. Try a different query or URL.",
                }
                if verbose:
                    print(f"  ✗ {tc.name}({tc.arguments}) [REPEATED — refused]")
            else:
                seen_actions.add(ah)
                tool_result = execute_tool(tc.name, tc.arguments)
                if verbose:
                    args_repr = (str(tc.arguments)[:80] + "...") if len(str(tc.arguments)) > 80 else str(tc.arguments)
                    print(f"  → {tc.name}({args_repr}) → {tool_result.get('status', '?')}")

                # Citations: only successful fetch_page calls
                if tc.name == "fetch_page" and tool_result.get("status") in ("ok", "too_long"):
                    citations.append({
                        "url": tool_result["url"],
                        "title": tool_result.get("title", ""),
                    })

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(tool_result)[:4000],
            })

    return {
        "answer": (
            f"I reached the step limit without a confident answer. "
            f"I fetched {len(citations)} page(s): "
            + ", ".join(c["url"] for c in citations[:3])
        ),
        "citations": citations,
        "steps": max_steps,
        "stopped_reason": "step_cap",
    }


## Demo

A medium-difficulty research query. The agent should issue 1-2
searches, fetch 1-3 pages, and produce an answer with citations
pointing only at the pages it actually read.

In [ ]:
result = run_agent(
    "What are the main differences between BM25 and dense retrieval for "
    "question answering? Cite specific sources."
)
print(f"\n=== Answer ===\n{result['answer']}")
print(f"\n=== Citations ({len(result['citations'])}) ===")
for c in result["citations"]:
    print(f"  • {c['title']}: {c['url']}")
print(f"\nSteps: {result['steps']}, stopped: {result['stopped_reason']}")


**Sample output (will vary — live web):**

```
── Step 1 ──
  → web_search({'query': 'BM25 vs dense retrieval question answering'}) → ok

── Step 2 ──
  → fetch_page({'url': 'https://example.com/...'}) → ok

── Step 3 ──
  → fetch_page({'url': 'https://...'}) → ok

── Step 4 ──
  ◆ FINAL: BM25 is a sparse term-matching retriever based on TF-IDF...

=== Answer ===
BM25 is a sparse lexical retriever (TF-IDF with length normalization)...

=== Citations (2) ===
  • Introduction to Information Retrieval — Chapter 11: ...
  • Dense Passage Retrieval — Karpukhin et al. 2020: ...

Steps: 4, stopped: answer_with_citations
```

## Production readiness — out of scope here

For a deployment add: a paid search backend with terms-of-service (Tavily,
Brave Search API, or Exa instead of `ddgs`), robots.txt awareness, per-host
politeness/throttling, response-size budgeting, content extraction beyond
bs4 (Trafilatura or Readability), and a citation-quality check before
synthesis. The agent loop above is the structural foundation; production
hardens its I/O layer.

See [`concepts/tools/search-tools.md`](../../../concepts/tools/search-tools.md)
for the search-vs-RAG distinction and
[`tools/search/snapshot-v1.0.md`](../../../tools/search/snapshot-v1.0.md)
for the current ddgs / Tavily / Brave tradeoffs.